In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from model import ScalingLaw, SampleAlpha
from constants import lower_bounds, test_models, delete_models, Y_names_tidy, Y_names, B, lrs, scheduler_factors, reps, n_epochs, random_seed
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ThreadPoolExecutor, as_completed
from sloth.sloth import Sloth

dims = [2, 3, 4, 5]
eps = 1e-3
Y_names = Y_names[0]

# To aggregate individual models
class JoinModels():
    def __init__(self, models):
        self.models = models
    
    def predict(self, X, D):
        Y_hat = np.hstack([model.predict(X, D) for model in self.models])
        return Y_hat

def to_latex(mu, ste, row_keys, col_keys, k_names, y_names,
             caption="", label="tab:results", fmt="{:.3f}", n_bold=3):
    """
    mu, ste: arrays of shape (n_rows, n_cols). If n_cols == len(col_keys)+1,
             the last column is treated as an aggregate ("Overall").
    Bolds the n_bold smallest mu values in each column.
    """
    n_cols = mu.shape[1]
    has_overall = (n_cols == len(col_keys) + 1)

    headers = [y_names[y] for y in col_keys] + (["Overall"] if has_overall else [])
    col_spec = "l" + " c" * n_cols

    # rank within each column; NaNs sent to the back so they're never bolded
    mu_ranked = np.where(np.isnan(mu), np.inf, mu)
    bold_mask = np.zeros_like(mu, dtype=bool)
    k = min(n_bold, mu.shape[0])
    for j in range(n_cols):
        idx = np.argsort(mu_ranked[:, j])[:k]
        bold_mask[idx, j] = True

    def cell(m, s, bold):
        m_str = fmt.format(m)
        inner = f"{m_str}_{{\\pm {fmt.format(s)}}}"
        if bold:
            inner = r"\boldsymbol{" + inner + "}"
        return f"${inner}$"
    
    lines = [
        r"\begin{table}[h]",
        r"\centering",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        " & " + " & ".join(headers) + r" \\",
        r"\midrule",
    ]

    row_keys = list(row_keys)
    for i, k_ in enumerate(row_keys):
        cells = [cell(mu[i, j], ste[i, j], bold_mask[i, j]) for j in range(n_cols)]
        lines.append(k_names[k_] + " & " + " & ".join(cells) + r" \\")
        if i < len(row_keys) - 1:
            lines.append(r"\midrule")

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\end{table}",
    ]
    return "\n".join(lines)

/Users/maiapolo/Desktop/statistical-scaling-law/scaling/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


Data

In [2]:
# Loading
df = pd.merge(pd.read_csv('data/df_full_v1.csv').drop('Unnamed: 0', axis=1),
              pd.read_csv('data/df_full_v2.csv').drop('Unnamed: 0', axis=1), 
              on=['model', 'family', 'size', 'tokens', 'flops'], how='outer')

# Creating data objects
Y = np.array(df.loc[:,Y_names])
Y = np.clip(Y, a_min=eps, a_max=1-eps)
        
X = np.log(np.array(df.loc[:,['size','tokens']]))
X = np.hstack((X,(X[:,0]*X[:,1])[:,None]))

F = np.array(df.loc[:,['size','tokens']])
F = np.log(F[:,0]*F[:,1]).reshape((-1,1))

D = np.array(pd.get_dummies(df.family)).astype(int)
I = np.ones(shape=(D.shape[0],1))
C = np.array([lower_bounds[s] for s in Y_names]).reshape((1,-1))

# Data split
test = []
for l in list(test_models.values()):
    test+=l
train = []
for l in list(delete_models.values()):
    train+=l

#0.084 -> 0.039
#train_idx = np.array([m not in delete_models[family] for m in df.model])
#test_idx = np.array([m in test_models[family] for m in df.model])
train_idx = np.array([m not in train for m in df.model])
test_idx = np.array([m in test for m in df.model])
test_models_list = list(df.model[test_idx])

X_train, F_train, Y_train, D_train, I_train = X[train_idx], F[train_idx], Y[train_idx], D[train_idx], I[train_idx]
X_test, F_test, Y_test, D_test, I_test = X[test_idx], F[test_idx], Y[test_idx], D[test_idx], I[test_idx]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
F_train = scaler.fit_transform(F_train)
F_test = scaler.transform(F_test)

Training (skip if already trained)

In [3]:
models = {}
predictions = {}

In [ ]:
# Unique intercept + FLOPs (Owen)
print("**** Unique intercept + FLOPs (Owen) ****")
ind_models = []
for j in tqdm(range(len(Y_names))):
    ind_models.append(Sloth(d=1))
    ind_models[-1].fit(F_train, I_train, Y_train[:,j:(j+1)], C[:,j:(j+1)], train_link=False, fit_C=True, positive_w=False, verbose=False, device='cpu')
models['flops'] = JoinModels(ind_models)
predictions['flops'] = models['flops'].predict(F_test, I_test)

# Unique intercept + Size/Tokens/Interaction
print("**** Unique intercept + Size/Tokens/Interaction ****")
ind_models = []
for j in tqdm(range(len(Y_names))):
    ind_models.append(Sloth(d=1))
    ind_models[-1].fit(X_train, I_train, Y_train[:,j:(j+1)], C[:,j:(j+1)], train_link=False, fit_C=True, positive_w=False, verbose=False, device='cpu')
models['size-tokens-inter'] = JoinModels(ind_models)
predictions['size-tokens-inter'] = models['size-tokens-inter'].predict(X_test, I_test)

In [6]:
# Family intercept + FLOPs
print("**** Family intercept + FLOPs****")
ind_models = []
for j in tqdm(range(len(Y_names))):
    ind_models.append(Sloth(d=1))
    ind_models[-1].fit(F_train, D_train, Y_train[:,j:(j+1)], C[:,j:(j+1)], train_link=False, fit_C=True, positive_w=False, verbose=False, device='cpu')
models['family-flops'] = JoinModels(ind_models)
predictions['family-flops'] = models['family-flops'].predict(F_test, D_test)

# Family intercept + Size/Tokens/Interaction
print("**** Family intercept + Size/Tokens/Interaction ****")
ind_models = []
for j in tqdm(range(len(Y_names))):
    ind_models.append(Sloth(d=1))
    ind_models[-1].fit(X_train, D_train, Y_train[:,j:(j+1)], C[:,j:(j+1)], train_link=False, fit_C=True, positive_w=False, verbose=False, device='cpu')
models['family-size-tokens-inter'] = JoinModels(ind_models)
predictions['family-size-tokens-inter'] = models['family-size-tokens-inter'].predict(X_test, D_test)

**** Family intercept + FLOPs****


  0%|          | 0/12 [00:00<?, ?it/s]

100%|██████████| 12/12 [01:26<00:00,  7.22s/it]


**** Family intercept + Size/Tokens/Interaction ****


100%|██████████| 12/12 [01:43<00:00,  8.65s/it]


In [ ]:
# Simple Sloth
print("**** Simple Sloth ****")
for dim in tqdm(dims, desc='dims'):
    models[f'simple-sloth_{dim}'] = Sloth(d=dim)
    models[f'simple-sloth_{dim}'].fit(X_train, D_train, Y_train, C, train_link=False, fit_C=False, positive_w=False, verbose=False, device='cpu')
    predictions[f'simple-sloth_{dim}'] = models[f'simple-sloth_{dim}'].predict(X_test, D_test)

# Sloth
print("**** Sloth ****")
for dim in tqdm(dims, desc='dims'):
    models[f'sloth_{dim}'] = Sloth(d=dim)
    models[f'sloth_{dim}'].fit(X_train, D_train, Y_train,
                               C0=C,
                               W1_X0=models[f'simple-sloth_{dim}'].W1_X.numpy(),
                               W1_D0=models[f'simple-sloth_{dim}'].W1_D.numpy(),
                               W20=models[f'simple-sloth_{dim}'].W2.numpy(),
                               b20=models[f'simple-sloth_{dim}'].b2.numpy(),
                               train_link=True, fit_C=True, positive_w=False, verbose=False, device='cpu')
    predictions[f'sloth_{dim}'] = models[f'sloth_{dim}'].predict(X_test, D_test)

In [ ]:
print("**** Ours ****")

def fit_one(dim, gpu_id):
    dev = f'cuda:{gpu_id}'
    with torch.cuda.device(gpu_id):              # pins this thread's default device
        m = ScalingLaw(dim)
        m.fit(X_train, Y_train, D_train, C,
              B=B, lrs=lrs,
              scheduler_factors=scheduler_factors,
              reps=reps, n_epochs=n_epochs,
              verbose=True,                     # interleaved logs from 4 threads = mess
              device=dev)
        y_hat = m.predict(X_train, Y_train, D_train, X_test, D_test, C)
    return dim, m, y_hat

with ThreadPoolExecutor(max_workers=len(dims)) as ex:
    futures = [ex.submit(fit_one, dim, i % torch.cuda.device_count())
               for i, dim in enumerate(dims)]
    for fut in tqdm(as_completed(futures), total=len(futures), desc='dims'):
        dim, m, y_hat = fut.result()
        models[f'ours_{dim}'] = m
        predictions[f'ours_{dim}'] = y_hat

In [ ]:
np.save(f"models/predictive_analysis/models.npy", models)
np.save(f"models/predictive_analysis/predictions.npy", predictions)

Results

In [8]:
models = np.load("models/predictive_analysis/models.npy", allow_pickle=True).item()
predictions = np.load("models/predictive_analysis/predictions.npy", allow_pickle=True).item()

In [13]:
k_names = {'flops':'FLOPs',
 'size-tokens-inter':'Size/Tokens/Interaction',
 'family-flops':'Family/FLOPs',
 'family-size-tokens-inter':'Family/Size/Tokens/Interaction',
 'simple-sloth_2':"Simple Sloth (d=2)",
 'simple-sloth_3':"Simple Sloth (d=3)",
 'simple-sloth_4':"Simple Sloth (d=4)",
 'simple-sloth_5':"Simple Sloth (d=5)",
 'sloth_2':"Sloth (d=2)",
 'sloth_3':"Sloth (d=3)",
 'sloth_4':"Sloth (d=4)",
 'sloth_5':"Sloth (d=5)",
 'ours_2':"Ours (d=2)",
 'ours_3':"Ours (d=3)",
 'ours_4':"Ours (d=4)",
 'ours_5':"Ours (d=5)"}

In [19]:
mu = []
ste = []

for k in k_names.keys():
    mu.append([])
    ste.append([])
    for j in range(len(Y_names)+1):
        if j == len(Y_names):
            e = np.abs(predictions[k]-Y_test)
        else:
            e = np.abs(predictions[k]-Y_test)[:,j]
        mask = ~np.isnan(e)
        
        mu[-1].append(e[mask].mean())
        ste[-1].append(e[mask].std()/np.sqrt(np.sum(mask)))

mu = np.array(mu)
ste = np.array(ste)  
Y_names_tidy["all"] = "All"

In [20]:
print(to_latex(mu, ste, list(k_names.keys()), Y_names+["all"], k_names, Y_names_tidy, n_bold=8))

\begin{table}[h]
\centering
\begin{tabular}{l c c c c c c c c c c c c c}
\toprule
 & MATH & IFEval & HellaSwag & BBH & MMLU-Pro & MMLU & ARC & TruthfulQA & GSM8k & Winogrande & GPQA & MuSR & All \\
\midrule
FLOPs & $0.078_{\pm 0.023}$ & $0.196_{\pm 0.066}$ & $0.025_{\pm 0.008}$ & $0.236_{\pm 0.042}$ & $\boldsymbol{0.084_{\pm 0.017}}$ & $0.091_{\pm 0.019}$ & $\boldsymbol{0.050_{\pm 0.008}}$ & $0.132_{\pm 0.042}$ & $0.113_{\pm 0.026}$ & $0.027_{\pm 0.006}$ & $0.222_{\pm 0.029}$ & $0.253_{\pm 0.034}$ & $0.141_{\pm 0.013}$ \\
\midrule
Size/Tokens/Interaction & $0.079_{\pm 0.023}$ & $0.195_{\pm 0.062}$ & $0.035_{\pm 0.010}$ & $0.117_{\pm 0.025}$ & $\boldsymbol{0.076_{\pm 0.016}}$ & $0.104_{\pm 0.024}$ & $0.062_{\pm 0.014}$ & $0.122_{\pm 0.034}$ & $0.174_{\pm 0.046}$ & $0.031_{\pm 0.007}$ & $0.128_{\pm 0.020}$ & $0.131_{\pm 0.023}$ & $0.109_{\pm 0.010}$ \\
\midrule
Family/FLOPs & $0.081_{\pm 0.032}$ & $\boldsymbol{0.057_{\pm 0.011}}$ & $\boldsymbol{0.018_{\pm 0.004}}$ & $\boldsymbol{0.051_{\

In [ ]:
list(predictions.keys())

['flops',
 'size-tokens-inter',
 'simple-sloth_2',
 'simple-sloth_3',
 'simple-sloth_4',
 'simple-sloth_5',
 'sloth_2',
 'sloth_3',
 'sloth_4',
 'sloth_5',
 'ours_2',
 'ours_3',
 'ours_4',
 'ours_5']